In [1]:
print("ram ram")

ram ram


In [2]:
import os
import uuid
import json
import base64
import io
import time
from pathlib import Path
from datetime import datetime
from PIL import Image
import fitz  # PyMuPDF
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
import ollama

# ====================== CONFIG ======================
INPUT_DIR = Path("input")
OUTPUT_IMAGES = Path("output/images")
OUTPUT_FINAL_MD = Path("output/final-md")
REPORT_FILE = Path("output/process_report.txt")

MODEL = "qwen2.5vl:3b"
MAX_IMAGE_SIZE = 768
USE_CPU = True
MAX_RETRIES = 3
# ====================================================

OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUTPUT_FINAL_MD.mkdir(parents=True, exist_ok=True)

def encode_image_resized(image_path: Path, max_size: int = 768) -> str:
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        return base64.b64encode(buffer.getvalue()).decode("utf-8")

def analyze_image_with_ollama(image_path: Path) -> dict:
    """Analyze image with retry logic"""
    image_b64 = encode_image_resized(image_path, MAX_IMAGE_SIZE)

    system_prompt = """You are an expert industrial document image analyst specialized in manufacturing equipment, HMI screens, and machine interfaces.
Return ONLY a valid JSON object following this exact schema:
{
  "asset_type": "embedded_document_image",
  "unique_image_id": "string",
  "source_file_origin": "string",
  "page_or_sheet_location": "string",
  "component_name": "string",
  "keywords": ["list", "of", "keywords"],
  "visual_specifications": {
    "expected_color": "detailed color description",
    "critical_alignment_angles": "layout description",
    "surface_texture_purity": "interface description"
  },
  "detailed_explanation": "Very detailed explanation of the image",
  "common_defects_and_failures": [
    {
      "defect_type": "string",
      "visual_signs": "string",
      "criticality_level": "HIGH / MEDIUM / LOW"
    }
  ],
  "local_storage_path": "string"
}
Rules:
- Return ONLY pure JSON. No markdown, no extra text.
- Be detailed and technical."""

    options = {"temperature": 0.1, "num_ctx": 2048}
    if USE_CPU:
        options["num_gpu"] = 0

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"    → Analyzing ({attempt}/{MAX_RETRIES}): {image_path.name}")
            response = ollama.chat(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": "Analyze this industrial image and return pure JSON.", "images": [image_b64]}
                ],
                format="json",
                options=options
            )
            result = json.loads(response["message"]["content"])
            result["local_storage_path"] = str(image_path).replace("\\", "/")
            return result
        except Exception as e:
            print(f"    [Retry {attempt}] Failed: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(2)
            else:
                return {
                    "error": str(e),
                    "local_storage_path": str(image_path).replace("\\", "/")
                }

def is_already_processed(file_stem: str) -> bool:
    """Skip if final MD already exists"""
    return (OUTPUT_FINAL_MD / f"{file_stem}.md").exists()

def extract_tables_from_page(page) -> str:
    """Extract tables from PDF page in clean Markdown"""
    tables = page.find_tables()
    if not tables:
        return ""
    
    md_tables = []
    for table in tables:
        data = table.extract()
        if not data:
            continue
        # Header
        header = " | ".join([str(cell or "").strip() for cell in data[0]])
        separator = " | ".join(["---"] * len(data[0]))
        rows = []
        for row in data[1:]:
            rows.append(" | ".join([str(cell or "").strip() for cell in row]))
        md_tables.append(f"| {header} |\n| {separator} |\n" + "\n".join([f"| {r} |" for r in rows]))
    return "\n\n".join(md_tables)

def process_pdf(pdf_path: Path, report: list):
    pdf_name = pdf_path.stem
    if is_already_processed(pdf_name):
        print(f"⏭️  Skipping (already processed): {pdf_path.name}")
        report.append(f"[SKIPPED] {pdf_path.name}")
        return

    print(f"\n{'='*60}")
    print(f"Processing PDF: {pdf_path.name}")
    print(f"{'='*60}")

    doc = fitz.open(pdf_path)
    img_dir = OUTPUT_IMAGES / pdf_name
    img_dir.mkdir(parents=True, exist_ok=True)

    md = [f"# {pdf_name}\n"]

    for page_num, page in enumerate(doc, start=1):
        md.append(f"\n## Page {page_num}\n")

        # Text
        text = page.get_text("text").strip()
        if text:
            md.append(text + "\n")

        # Tables
        tables_md = extract_tables_from_page(page)
        if tables_md:
            md.append("\n### Tables\n")
            md.append(tables_md + "\n")

        # Images
        for img in page.get_images(full=True):
            xref = img[0]
            base = doc.extract_image(xref)
            ext = base["ext"]
            unique = uuid.uuid4().hex[:8]
            img_path = img_dir / f"{unique}.{ext}"

            with open(img_path, "wb") as f:
                f.write(base["image"])
            print(f"  → Saved: {img_path.name}")

            analysis = analyze_image_with_ollama(img_path)
            analysis["page_or_sheet_location"] = f"Page {page_num}"

            md.append("\n### Image Analysis\n")
            md.append("```json")
            md.append(json.dumps(analysis, indent=2, ensure_ascii=False))
            md.append("```\n")

    doc.close()

    final_path = OUTPUT_FINAL_MD / f"{pdf_name}.md"
    final_path.write_text("\n".join(md).strip() + "\n", encoding="utf-8")
    print(f"✅ Final MD saved: {final_path}")
    report.append(f"[SUCCESS] {pdf_path.name}")

def process_ppt(ppt_path: Path, report: list):
    ppt_name = ppt_path.stem
    if is_already_processed(ppt_name):
        print(f"⏭️  Skipping (already processed): {ppt_path.name}")
        report.append(f"[SKIPPED] {ppt_path.name}")
        return

    if ppt_path.suffix.lower() == ".ppt":
        print(f"⚠️  Skipping old .ppt format: {ppt_path.name}")
        report.append(f"[SKIPPED - OLD FORMAT] {ppt_path.name}")
        return

    print(f"\n{'='*60}")
    print(f"Processing PPT: {ppt_path.name}")
    print(f"{'='*60}")

    prs = Presentation(ppt_path)
    img_dir = OUTPUT_IMAGES / ppt_name
    img_dir.mkdir(parents=True, exist_ok=True)

    md = [f"# {ppt_name}\n"]

    for slide_num, slide in enumerate(prs.slides, start=1):
        md.append(f"\n## Slide {slide_num}\n")

        # Text
        texts = []
        for shape in slide.shapes:
            if shape.has_text_frame:
                for para in shape.text_frame.paragraphs:
                    line = "".join([run.text for run in para.runs]).strip()
                    if line:
                        texts.append(line)
        if texts:
            md.append("\n".join(texts) + "\n")

        # Images
        for shape in slide.shapes:
            if shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
                try:
                    image = shape.image
                    ext = image.ext
                    unique = uuid.uuid4().hex[:8]
                    img_path = img_dir / f"{unique}.{ext}"

                    with open(img_path, "wb") as f:
                        f.write(image.blob)
                    print(f"  → Saved: {img_path.name}")

                    analysis = analyze_image_with_ollama(img_path)
                    analysis["page_or_sheet_location"] = f"Slide {slide_num}"

                    md.append("\n### Image Analysis\n")
                    md.append("```json")
                    md.append(json.dumps(analysis, indent=2, ensure_ascii=False))
                    md.append("```\n")
                except Exception as e:
                    print(f"  [ERROR] Image extract failed: {e}")

    final_path = OUTPUT_FINAL_MD / f"{ppt_name}.md"
    final_path.write_text("\n".join(md).strip() + "\n", encoding="utf-8")
    print(f"✅ Final MD saved: {final_path}")
    report.append(f"[SUCCESS] {ppt_path.name}")

# ====================== MAIN ======================
if __name__ == "__main__":
    start_time = datetime.now()
    report = []
    report.append(f"Process started: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    report.append("=" * 60)

    # Collect files
    pdf_files = list(INPUT_DIR.glob("*.pdf"))
    ppt_files = list(INPUT_DIR.glob("*.pptx")) + list(INPUT_DIR.glob("*.ppt"))

    print(f"Found {len(pdf_files)} PDF(s) and {len(ppt_files)} PPT(s)\n")

    for pdf in pdf_files:
        process_pdf(pdf, report)

    for ppt in ppt_files:
        process_ppt(ppt, report)

    end_time = datetime.now()
    report.append("=" * 60)
    report.append(f"Process finished: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
    report.append(f"Total time: {end_time - start_time}")

    # Save report
    REPORT_FILE.write_text("\n".join(report), encoding="utf-8")
    print(f"\n📄 Report saved → {REPORT_FILE}")
    print("\n".join(report))

Found 1 PDF(s) and 2 PPT(s)

⏭️  Skipping (already processed): Teeth Surface Forming Machine.pdf
⏭️  Skipping (already processed): C Ring pressing NG - Yokoten management reflection.pptx
⏭️  Skipping (already processed): Worm Shaft Crack Updated QIP (003).pptx

📄 Report saved → output\process_report.txt
Process started: 2026-09-29 17:25:43
[SKIPPED] Teeth Surface Forming Machine.pdf
[SKIPPED] C Ring pressing NG - Yokoten management reflection.pptx
[SKIPPED] Worm Shaft Crack Updated QIP (003).pptx
Process finished: 2026-09-29 17:25:43
Total time: 0:00:00.019257
